In [ ]:
print("TFC_Spline  - Maryam Yaghobi , Dr Jafari, Dr Hosseini")
print("Example_1_1.2 \n")

import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import math as mt
import time
import matplotlib.pyplot as plt

import platform
import psutil
import sys
import os

import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


hidden = 40
print("NN: (2 + 2*", hidden, " + 1)\n")
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(2, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, hidden)
        self.fc4 = nn.Linear(hidden, hidden)
        self.fc5 = nn.Linear(hidden, 1)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        #x = torch.tanh(self.fc3(x))
        #x = torch.tanh(self.fc4(x))
        x = self.fc5(x)
        return x

model = Net().to(device)
start_time = time.time()

n = 40;        dt = 1.0/n;   print("n =",n)
alpha = torch.tensor(1.2, dtype=torch.float64, device=device);print("alpha=",alpha)

n_test =100
n_test_less = 17

tt = torch.linspace(0, 1, n+1)
xx = torch.linspace(0, 1, n+1)
t_grid, x_grid = torch.meshgrid(tt, xx, indexing='ij')
t_train = t_grid.reshape(-1, 1).to(device)
x_train = x_grid.reshape(-1, 1).to(device)
t_train.requires_grad_(True)
x_train.requires_grad_(True)

x_b = torch.linspace(0, 1, n+1, dtype=torch.float64, device=device).reshape(-1,1)
x_b.requires_grad_(True)

t0 = torch.zeros_like(x_b,dtype=torch.float64, device=device)
t0.requires_grad_(True)

t1 = torch.ones_like(x_b,dtype=torch.float64, device=device)
t1.requires_grad_(True)

def e(num):   #Convert to sci form
    return (f'{num:.3e}')

def exact_y(t,x):
    y = x*t
    return y

y_exact = exact_y(t_train, x_train)

def gamma(x):
    x = torch.as_tensor(x, dtype=torch.float64, device=device)
    return torch.exp(torch.lgamma(x))


def test(nt):
    tt_test = torch.linspace(0, 1, nt)
    xx_test = torch.linspace(0, 1, nt)

    tt_test = tt_test.to(dtype=torch.float64, device=device)
    xx_test = xx_test.to(dtype=torch.float64, device=device)

    t_grid_test, x_grid_test = torch.meshgrid(tt_test, xx_test, indexing='ij')

    t_test = t_grid_test.reshape(-1, 1).to(device)
    x_test = x_grid_test.reshape(-1, 1).to(device)

    t_test.requires_grad_(True)
    x_test.requires_grad_(True)

    model.eval()

    g_test = model(torch.cat((t_test, x_test), dim=1))
    g_t_test = torch.autograd.grad(g_test.sum(), t_test, create_graph=True)[0]
    y_test = g_test + H(t_test, x_test, nt)
    y_exact_test = exact_y(t_test, x_test)

    error_test = y_test - y_exact_test

    abs_error = torch.abs(error_test)
    rmse_test = torch.sqrt(error_test.pow(2).mean()).item()
    rl2_test = (torch.sqrt(error_test.pow(2).sum())/torch.sqrt(y_exact_test.pow(2).sum())).item()
    abs_error_test = abs_error.max().item()
    mae_test = abs_error.mean().item()

    # Location of maximum error (new)
    idx_max = torch.argmax(abs_error)

    peak_t = t_test[idx_max].item()
    peak_x = x_test[idx_max].item()

    return (rmse_test, rl2_test, abs_error_test, mae_test,     y_test, peak_t, peak_x )

def time_derivative(u, t_var):
    return torch.autograd.grad(outputs=u,inputs=t_var, grad_outputs=torch.ones_like(u), create_graph=True)[0]

def H(t, x,num):

    c00_t = 0.0
    c01_t = 1.0
    c00 = 0.0
    c01 = 0.0

    Yt0 = torch.zeros_like(t)
    Yt1 = t
    Y0x = torch.zeros_like(x)
    Y0x_t = x

    x0 = torch.zeros_like(x)
    x1 = torch.ones_like(x)
    t0 = torch.zeros_like(t, requires_grad=True)

    g_t0 = model(torch.cat((t,  x0), dim=1))
    g_t1 = model(torch.cat((t,  x1), dim=1))
    g_0x = model(torch.cat((t0, x), dim=1))

    g_00 = model(torch.cat((t0, x0), dim=1))
    g_01 = model(torch.cat((t0, x1), dim=1))

    g_0x_t = time_derivative(g_0x, t0)
    g_00_t = time_derivative(g_00, t0)
    g_01_t = time_derivative(g_01, t0)

    d_t0 = Yt0 - g_t0
    d_t1 = Yt1 - g_t1
    d_0x = Y0x - g_0x
    d_0x_t = Y0x_t - g_0x_t

    d_00 = c00 - g_00
    d_01 = c01 - g_01
    d_00_t = c00_t - g_00_t
    d_01_t = c01_t - g_01_t

    s1 = d_0x + t * d_0x_t
    s2 = d_t0 - d_00 - t * d_00_t
    s3 = d_t1 - d_01 - t * d_01_t

    return s1 + (1.0 - x) * s2 + x * s3

def yCE(t,x,g):
    y = g + H(t, x , n+1)
    return y

def F(t ,x ,y):
    y_x = torch.autograd.grad(y.sum(), x, create_graph=True)[0]
    y_xx = torch.autograd.grad(y_x.sum(), x, create_graph=True)[0]
    out1 = y_xx+ y**2 - x**2*t**2
    return  out1.reshape(n+1,n+1)


precompute_start = time.time()

R = 1 / gamma(alpha)

t2 = torch.zeros(n+1,n+1).to(device)
Tk = torch.zeros(2,n+1,n).to(device)

for i in range(n+1):
    for j in range(n+1):
        t2[i,j] = tt[i]-tt[j]

for m in range(2):
    for k in range(n+1):
        for j in range(n):
            Tk[m,k,j] = (t2[k,j+1]**(alpha+m)-t2[k,j]**(alpha+m))/(alpha+m)

precompute_end = time.time()
precompute_time = precompute_end - precompute_start

def loss_function(t ,x ,y):
    Func = F(t ,x ,y).reshape(n+1,n+1)
    Fj0 = torch.zeros(n,n+1)
    Fj1 = torch.zeros(n,n+1)

    for j in range(0,n):
      Fj0[j]= Func[j,:]   / t2[j,j+1]
      Fj1[j]= Func[j+1,:] / t2[j,j+1]

    L = torch.zeros_like(Func)
    #L = g # #(y[t==tt[0]]-Y0).pow(2) 0.0  #
    for i in range(1,n+1):
      sigma= torch.zeros(n+1)#.to(device)
      for j in range(0,i):
        sigma += Fj0[j] *(  Tk[1,i,j] - t2[i,j+1]*Tk[0,i,j] )\
                -Fj1[j] *(  Tk[1,i,j] - t2[i,j]  *Tk[0,i,j] )

      L[i,:] = y.reshape(n+1,n+1)[i,:] -R*sigma

    L= L - ( x * t).reshape(n+1, n+1)

    return L .pow(2) .mean()


epochs = 30000 ;print("Max epochs =",epochs,"\n")

optimizer = optim.Adam(model.parameters(),lr=0.001)
patience = 500
min_delta = 1e-14
best_loss = float('inf')

train_start = time.time()
counter = 0

loss_history = []

train_rmse_history = []
train_rl2_history = []
train_abs_history = []
train_mae_history = []

test_rmse_history  = []
test_rl2_history  = []
test_abs_history  = []
test_mae_history  = []

#last_epoch = 0

for epoch in range(0,epochs+1):

    model.train()
    optimizer.zero_grad()

    g = model(torch.cat((t_train, x_train), dim=1))

    y = yCE(t_train, x_train ,g)

    loss  = loss_function(t_train, x_train ,y)

    loss.backward()
    optimizer.step()

    last_epoch = epoch

    loss_history.append(loss.item())

    error_train = y - exact_y(t_train, x_train)
    RMSE = torch.sqrt(error_train.pow(2).mean()).item()
    RL2 = (torch.sqrt(error_train.pow(2).sum())/torch.sqrt(exact_y(t_train, x_train).pow(2).sum())).item()
    ABS = torch.abs(error_train).max().item()
    MAE = torch.abs(error_train).mean().item()
    rmse_test, rl2_test, abs_test, mae_test, _, peak_t, peak_x = test(n_test)

    train_rmse_history.append(RMSE)
    train_rl2_history.append(RL2)
    train_abs_history.append(ABS)
    train_mae_history.append(MAE)

    test_rmse_history.append(rmse_test)
    test_rl2_history.append(rl2_test)
    test_abs_history.append(abs_test)
    test_mae_history.append(mae_test)

    if (epoch) % 100 == 0:
        with torch.no_grad():
            rmse = torch.sqrt((y - exact_y(t_train, x_train)).pow(2).mean())
            RL2 = torch.sqrt((y - exact_y(t_train, x_train)).pow(2).sum()) / torch.sqrt(exact_y(t_train, x_train).pow(2).sum())
            ABS = torch.abs((y - exact_y(t_train,x_train))).max()
            #test_metrics = test()
        print(f'{epoch:5} #', e(loss.item()),"| RMSE:", e(rmse),"  RL2:",e(RL2),"  ABS:",e(ABS)," MAE:", e(MAE),
        "  | test:  rmse:", e(rmse_test),"  rl2:", e(rl2_test),"  abs:", e(abs_test)," mae:", e(mae_test))



model.eval()
g_pred = model(torch.cat((t_train, x_train), dim=1))
y_pred = g_pred + H(t_train, x_train, n+1)

with torch.no_grad():
    #y = y_pred
    rmse = torch.sqrt((y - exact_y(t_train, x_train)).pow(2).mean())
    RL2 = torch.sqrt((y - exact_y(t_train, x_train)).pow(2).sum()) / torch.sqrt(exact_y(t_train, x_train).pow(2).sum())
    ABS = torch.abs((y - exact_y(t_train,x_train))).max()
print("\n",f'{epoch:5} #', e(loss.item()),"| RMSE:", e(rmse),"  RL2:",e(RL2),"  ABS:",e(ABS)," MAE:", e(MAE),
"  | test:  rmse:", e(rmse_test),"  rl2:", e(rl2_test),"  abs:", e(abs_test)," mae:", e(mae_test),"\n")

train_end = time.time()
train_time = train_end - train_start

process = psutil.Process(os.getpid())
memory_MB = process.memory_info().rss /1024**2

print("corner values")
print("u(0,0)=",y_pred.reshape(n+1, n+1)[0,0].item())
print("exact_00=",y_exact.reshape(n+1, n+1)[0,0].item())

print("u(0,1)=",y_pred.reshape(n+1, n+1)[0,-1].item())
print("exact_01=",y_exact.reshape(n+1, n+1)[0,-1].item())

print("u(1,0)=",y_pred.reshape(n+1, n+1)[-1,0].item())
print("exact_10=",y_exact.reshape(n+1, n+1)[-1,0].item())

print("u(1,1)=",y_pred.reshape(n+1, n+1)[-1,-1].item())
print("exact_11=",y_exact.reshape(n+1, n+1)[-1,-1].item())

print("\nTesting:")

test_start = time.time()

errorL2_test =(test(n_test)[1])# 100*100

test_end = time.time()
test_time = test_end - test_start
print("Relative L2 Error on ",f'{n_test:3}',"x",f'{n_test:3}'," grid:",e(errorL2_test))

errorL2_test =(test(n_test_less)[1]) # 17*17
print("Relative L2 Error on ",f'{n_test_less:3}',"x",f'{n_test_less:3}'," grid:",e(errorL2_test))


g = g_pred.detach().cpu().numpy().reshape(n + 1, n + 1)
yCE = y_pred.detach().cpu().numpy().reshape(n + 1, n + 1)
y_pred_2d = yCE.reshape(n + 1, n + 1)
error_2d = np.abs(y_pred.detach().cpu().numpy().reshape(n+1, n+1) -
           exact_y(t_grid.reshape(-1, 1), x_grid.reshape(-1, 1)).cpu().numpy().reshape(n+1, n+1))

fig = plt.figure(figsize=(15,5))
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(t_grid.numpy(), x_grid.numpy(), y_pred_2d, cmap='viridis')
ax1.set_title('Approx y')

exact_y_2d = exact_y(t_grid.reshape(-1, 1), x_grid.reshape(-1, 1)).cpu().numpy().reshape(n+1, n+1)
ax2 = fig.add_subplot(132, projection='3d')
ax2.plot_surface(t_grid.numpy(), x_grid.numpy(), exact_y_2d, cmap='viridis')
ax2.set_title('Exact y')

ax3 = fig.add_subplot(133, projection='3d')
ax3.plot_surface(t_grid.numpy(), x_grid.numpy(), error_2d, cmap='inferno')
ax3.set_title('Absolute Error |y_pred - y_exact|')
plt.show()

ax4 = fig.add_subplot(122, projection='3d')
ax4.plot_surface(t_grid.numpy(), x_grid.numpy(), g, cmap='viridis')
ax4.set_title('Approx g')
plt.show()

plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Function')
plt.show()

model.eval()

rmse_test, rl2_test, abs_test, mae_test, y_test, peak_t, peak_x = test(n_test)

tt_test = torch.linspace(0, 1, n_test)
xx_test = torch.linspace(0, 1, n_test)
t_grid_test, x_grid_test = torch.meshgrid(tt_test, xx_test, indexing='ij')

y_test_np = y_test.detach().cpu().numpy().reshape(n_test, n_test)
t_grid_test_np = t_grid_test.cpu().numpy()
x_grid_test_np = x_grid_test.cpu().numpy()

y_exact_test_np = exact_y(t_grid_test.reshape(-1,1), x_grid_test.reshape(-1,1)).cpu().numpy().reshape(n_test, n_test)

error_test_np = np.abs(y_test_np - y_exact_test_np)

fig = plt.figure(figsize=(15,5))
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(t_grid_test_np, x_grid_test_np, y_test_np, rstride=1, cstride=1, cmap='viridis')
ax1.set_title('Approx y (test)')
ax1.set_xlabel('t'); ax1.set_ylabel('x')

ax2 = fig.add_subplot(132, projection='3d')
ax2.plot_surface(t_grid_test_np, x_grid_test_np, y_exact_test_np, rstride=1, cstride=1, cmap='viridis')
ax2.set_title('Exact y (test)')
ax2.set_xlabel('t'); ax2.set_ylabel('x')

ax3 = fig.add_subplot(133, projection='3d')
ax3.plot_surface(t_grid_test_np, x_grid_test_np, error_test_np, rstride=1, cstride=1, cmap='inferno')
ax3.set_title('Absolute Error |y_test - y_exact|')
ax3.set_xlabel('t'); ax3.set_ylabel('x')

plt.tight_layout()
plt.show()

plt.figure(figsize=(20,5))
plt.subplot(1,2,1)
plt.plot(train_rmse_history, label='Train RMSE')
plt.plot(test_rmse_history, label='Test RMSE')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('MSE (log scale)')
plt.title('Train/Test MSE')
plt.legend()

#plt.subplot(figsize=(10,5))
plt.subplot(1,2,2)
plt.plot(train_abs_history, label='Train MSE')
plt.plot(test_abs_history, label='Test MSE')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('ABS (log scale)')
plt.title('Train/Test MSE')
plt.legend()

plt.figure(figsize=(15,5))
plt.subplot(1,2,1)
plt.plot(train_rl2_history, label='Train RL2')
plt.plot(test_rl2_history, label='Test RL2')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Relative L2 (log scale)')
plt.title('Train/Test RL2')
plt.legend()

plt.tight_layout()
plt.show()


y_test_np = y_test.detach().cpu().numpy().reshape(n_test, n_test)
y_exact_test_np = exact_y(t_grid_test.reshape(-1,1), x_grid_test.reshape(-1,1)).cpu().numpy().reshape(n_test, n_test)
error_test_np = np.abs(y_test_np - y_exact_test_np)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im1 = axes[0].imshow(y_test_np, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='viridis')
axes[0].set_title("Approx y (test)")
axes[0].set_xlabel("x"); axes[0].set_ylabel("t")
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(y_exact_test_np, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='viridis')
axes[1].set_title("Exact y (test)")
axes[1].set_xlabel("x"); axes[1].set_ylabel("t")
plt.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow(error_test_np, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='inferno')
axes[2].set_title("Absolute Error (test)")
axes[2].set_xlabel("x"); axes[2].set_ylabel("t")
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.savefig("Exact y(test)_Approx y(test)_Absolute Error(test).pdf")

plt.show()


plt.figure(figsize=(8,5))
plt.plot(train_rl2_history, label='Train RL2')
plt.plot(test_rl2_history, label='Test RL2')
plt.yscale('log')
plt.xlabel('Epoch'); plt.ylabel('Relative L2 (log scale)')
plt.title('Train/Test RL2')
plt.legend()
plt.tight_layout()
plt.savefig("Train_Test_RL2.pdf")
plt.show()

pdf_files = [
    #"Exact y(train)_Approx y(train)_Absolute Error(train).pdf",
    "Exact y(test)_Approx y(test)_Absolute Error(test).pdf",
    "Train_Test_RL2.pdf"
]
from google.colab import files
for f in pdf_files:
    files.download(f)



In [ ]:
print("TFC_Hermite  - Maryam Yaghobi , Dr Jafari Dr Hosseini")
print("Example_1_TFC_Hermite_e^-3 \n")

import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import math as mt
import time
import matplotlib.pyplot as plt

import platform
import psutil
import sys
import os

import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


hidden = 40
print("NN: (2 + 3*", hidden, " + 1)\n")
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(2, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, hidden)
        self.fc4 = nn.Linear(hidden, hidden)
        self.fc5 = nn.Linear(hidden, 1)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        #x = torch.tanh(self.fc3(x))
        #x = torch.tanh(self.fc4(x))
        x = self.fc5(x)
        return x

model = Net().to(device)
start_time = time.time()

n = 40;        dt = 1.0/n;   print("n =",n)
alpha = torch.tensor(1.2, dtype=torch.float64, device=device);print("alpha=",alpha)

n_test =100
n_test_less = 17

tt = torch.linspace(0, 1, n+1)
xx = torch.linspace(0, 1, n+1)
t_grid, x_grid = torch.meshgrid(tt, xx, indexing='ij')
t_train = t_grid.reshape(-1, 1).to(device)
x_train = x_grid.reshape(-1, 1).to(device)
t_train.requires_grad_(True)
x_train.requires_grad_(True)

x_b = torch.linspace(0, 1, n+1, dtype=torch.float64, device=device).reshape(-1,1)
x_b.requires_grad_(True)
t0 = torch.zeros_like(x_b, device=device)
t0.requires_grad_(True)

t1 = torch.ones_like(x_b, device=device)
t1.requires_grad_(True)

def e(num):
    return (f'{num:.3e}')

def exact_y(t,x):
    y = x*t
    return y

y_exact = exact_y(t_train, x_train)

def gamma(x):
    x = torch.as_tensor(x, dtype=torch.float64, device=device)
    return torch.exp(torch.lgamma(x))


def test(nt):
    tt_test = torch.linspace(0, 1, nt)
    xx_test = torch.linspace(0, 1, nt)

    tt_test = tt_test.to(dtype=torch.float64, device=device)
    xx_test = xx_test.to(dtype=torch.float64, device=device)

    t_grid_test, x_grid_test = torch.meshgrid(tt_test, xx_test, indexing='ij')

    t_test = t_grid_test.reshape(-1, 1).to(device)
    x_test = x_grid_test.reshape(-1, 1).to(device)

    t_test.requires_grad_(True)
    x_test.requires_grad_(True)

    model.eval()

    g_test = model(torch.cat((t_test, x_test), dim=1))
    g_t_test = torch.autograd.grad(g_test.sum(), t_test, create_graph=True)[0]
    y_test = g_test + H(t_test, x_test, nt)
    y_exact_test = exact_y(t_test, x_test)

    error_test = y_test - y_exact_test

    abs_error = torch.abs(error_test)
    rmse_test = torch.sqrt(error_test.pow(2).mean()).item()
    rl2_test = (torch.sqrt(error_test.pow(2).sum())/torch.sqrt(y_exact_test.pow(2).sum())).item()
    abs_error_test = abs_error.max().item()
    mae_test = abs_error.mean().item()

    # Location of maximum error (new)
    idx_max = torch.argmax(abs_error)

    peak_t = t_test[idx_max].item()
    peak_x = x_test[idx_max].item()

    return (rmse_test, rl2_test, abs_error_test, mae_test,     y_test, peak_t, peak_x )

y0x = torch.zeros_like(xx)

def time_derivative(u, t_var):
    return torch.autograd.grad(outputs=u,inputs=t_var, grad_outputs=torch.ones_like(u), create_graph=True)[0]

def H(t, x,num):

    c00_t = 0.0
    c01_t = 1.0
    c00 = 0.0
    c01 = 0.0

    Yt0 = torch.zeros_like(t)
    Yt1 = t
    Y0x = torch.zeros_like(x)
    Y0x_t = x

    x0 = torch.zeros_like(x)
    x1 = torch.ones_like(x)
    t0 = torch.zeros_like(t, requires_grad=True)

    g_t0 = model(torch.cat((t,  x0), dim=1))
    g_t1 = model(torch.cat((t,  x1), dim=1))
    g_0x = model(torch.cat((t0, x), dim=1))

    g_00 = model(torch.cat((t0, x0), dim=1))
    g_01 = model(torch.cat((t0, x1), dim=1))

    g_0x_t = time_derivative(g_0x, t0)
    g_00_t = time_derivative(g_00, t0)
    g_01_t = time_derivative(g_01, t0)

    d_t0 = Yt0 - g_t0
    d_t1 = Yt1 - g_t1
    d_0x = Y0x - g_0x
    d_0x_t = Y0x_t - g_0x_t

    d_00 = c00 - g_00
    d_01 = c01 - g_01
    d_00_t = c00_t - g_00_t
    d_01_t = c01_t - g_01_t

    s1 = d_0x + t * d_0x_t
    s2 = d_t0 - d_00 - t * d_00_t
    s3 = d_t1 - d_01 - t * d_01_t

    return s1 + (1.0 - x) * s2 + x * s3


def yCE(t,x,g):
    y = g + H(t, x , n+1)
    return y

def F_and_Ft(t ,x ,y):
    y_x = torch.autograd.grad(y.sum(), x, create_graph=True)[0]
    y_xx = torch.autograd.grad(y_x.sum(), x, create_graph=True)[0]
    y_xx_t = torch.autograd.grad(y_xx.sum(), x, create_graph=True)[0]

    out1 = y_xx - y**2 + (x**2)*(t**2)
    out2 = y_xx_t - 2*y*y_x + 2*x*(t**2)

    return  out1.reshape(n+1,n+1)    ,out2.reshape(n+1,n+1)

precompute_start = time.time()

R = 1 /gamma(alpha)


t2 = tt.unsqueeze(1) - tt.unsqueeze(0)

M =  -t2[:,0:n]

N0 = t2[:,0:n]
N1 = t2[:,1:n+1]


precompute_end = time.time()
precompute_time = precompute_end - precompute_start
y0x   = 0.0
y0x_t = xx

def loss_function(t ,x ,g ,y):
    Func,Func_t = F_and_Ft(t ,x , y)

    L = torch.zeros(n+1, n+1, dtype=Func.dtype, device=Func.device)

    for i in range(1,n+1):
        j = torch.arange(0, i)

        N0j = N0[i,j].unsqueeze(1).to(Func.device)
        N1j = N1[i,j].unsqueeze(1).to(Func.device)
        Mj  = M[i,j].unsqueeze(1).to(Func.device)

        a = Func[j,:]
        b = Func_t[j,:]
        c = (3*(Func[j+1,:]-Func[j,:])/dt**2  -  (2*Func_t[j,:]+Func_t[j+1,:])/dt)
        d = (2*(Func[j,:]-Func[j+1,:])/dt**3  +  (Func_t[j,:]+Func_t[j+1,:])/dt**2)

        p = d /(alpha+3.0)
        q = (3*(Mj*d)-c) /(alpha+2.0)
        r = (3*(Mj**2)*d -2*(Mj*c) +b) /(alpha+1.0)
        s = ((Mj**3)*d -(Mj**2)*c +(Mj*b) - a) /alpha

        term_1 = N0j**(alpha+3)*p + N0j**(alpha+2)*q + N0j**(alpha+1)*r + N0j**(alpha)*s
        term_2 = N1j**(alpha+3)*p + N1j**(alpha+2)*q + N1j**(alpha+1)*r + N1j**(alpha)*s

        sigma = (term_2 - term_1).sum(dim=0)

        L[i,:] =  y.reshape(n+1, n+1)[i,:] - R*sigma
    L= L - ( x * t).reshape(n+1, n+1)

    return L.pow(2) .mean()


epochs = 30000  ;print("Max epochs =",epochs,"\n")

optimizer = optim.Adam(model.parameters(),lr=0.001)
patience = 500
#min_delta = 1e-14
best_loss = float('inf')

train_start = time.time()
counter = 0

loss_history = []

train_rmse_history = []
train_rl2_history = []
train_abs_history = []
train_mae_history = []

test_rmse_history  = []
test_rl2_history  = []
test_abs_history  = []
test_mae_history  = []



for epoch in range(0,epochs+1):

    model.train()
    optimizer.zero_grad()

    g = model(torch.cat((t_train, x_train), dim=1))

    y = yCE(t_train, x_train ,g)

    loss  = loss_function(t_train, x_train ,g ,y)

    loss.backward()
    optimizer.step()

    last_epoch = epoch

    loss_history.append(loss.item())

    error_train = y - exact_y(t_train, x_train)
    RMSE = torch.sqrt(error_train.pow(2).mean()).item()
    RL2 = (torch.sqrt(error_train.pow(2).sum())/torch.sqrt(exact_y(t_train, x_train).pow(2).sum())).item()
    ABS = torch.abs(error_train).max().item()
    MAE = torch.abs(error_train).mean().item()
    rmse_test, rl2_test, abs_test, mae_test, _, peak_t, peak_x = test(n_test)

    train_rmse_history.append(RMSE)
    train_rl2_history.append(RL2)
    train_abs_history.append(ABS)
    train_mae_history.append(MAE)

    test_rmse_history.append(rmse_test)
    test_rl2_history.append(rl2_test)
    test_abs_history.append(abs_test)
    test_mae_history.append(mae_test)

    if (epoch) % 200 == 0:
        with torch.no_grad():
            rmse = torch.sqrt((y - exact_y(t_train, x_train)).pow(2).mean())
            RL2 = torch.sqrt((y - exact_y(t_train, x_train)).pow(2).sum()) / torch.sqrt(exact_y(t_train, x_train).pow(2).sum())
            ABS = torch.abs((y - exact_y(t_train,x_train))).max()
            #test_metrics = test()
        print(f'{epoch:5} #', e(loss.item()),"| RMSE:", e(rmse),"  RL2:",e(RL2),"  ABS:",e(ABS)," MAE:", e(MAE),
        "  | test:  rmse:", e(rmse_test),"  rl2:", e(rl2_test),"  abs:", e(abs_test)," mae:", e(mae_test))


model.eval()
g_pred = model(torch.cat((t_train, x_train), dim=1))
g_t_pred = torch.autograd.grad(g_pred.sum(), t_train, create_graph=True)[0]
y_pred = g_pred + H(t_train, x_train, n+1)

train_end = time.time()
train_time = train_end - train_start

process = psutil.Process(os.getpid())
memory_MB = process.memory_info().rss /1024**2

print("corner values")
print("u(0,0)=",y_pred[0,0].item())
print("exact_00=",y_exact[0,0].item())

print("u(0,1)=",y_pred[0,-1].item())
print("exact_01=",y_exact[0,-1].item())

print("u(1,0)=",y_pred[-1,0].item())
print("exact_10=",y_exact[-1,0].item())

print("u(1,1)=",y_pred[-1,-1].item())
print("exact_11=",y_exact[-1,-1].item())

print("\nTesting:")

test_start = time.time()

tt_test = torch.linspace(0, 1, n_test).requires_grad_(True)
xx_test = torch.linspace(0, 1, n_test)

tt_test = tt_test.to(dtype=torch.float64)
xx_test = xx_test.to(dtype=torch.float64)

t_grid_test, x_grid_test = torch.meshgrid(tt_test, xx_test, indexing='ij')
t_test = t_grid_test.reshape(-1, 1)
x_test = x_grid_test.reshape(-1, 1)

device = next(model.parameters()).device
t_test = t_test.to(device)
x_test = x_test.to(device)

y_exact_test = exact_y(t_test, x_test)
g_pred_test = model(torch.cat([t_test, x_test], dim=1))[:, 0].reshape(-1, 1)

y_pred_test = g_pred_test + H(t_test, x_test, n_test)
y_exact_test = exact_y(t_test, x_test)

test_end = time.time()
test_time = test_end - test_start

errorL2_test = torch.sqrt((y_pred_test - y_exact_test).pow(2).sum()) / torch.sqrt(y_exact_test.pow(2).sum())
print("Relative L2 Error on ",n_test_less,"x",n_test_less," grid:",e(errorL2_test.item()))


tt_test = torch.linspace(0, 1, n_test_less).requires_grad_(True)
xx_test = torch.linspace(0, 1, n_test_less)

tt_test = tt_test.to(dtype=torch.float64)
xx_test = xx_test.to(dtype=torch.float64)

t_grid_test, x_grid_test = torch.meshgrid(tt_test, xx_test, indexing='ij')
t_test = t_grid_test.reshape(-1, 1)
x_test = x_grid_test.reshape(-1, 1)

device = next(model.parameters()).device
t_test = t_test.to(device)
x_test = x_test.to(device)

y_exact_test = exact_y(t_test, x_test)
g_pred_test = model(torch.cat([t_test, x_test], dim=1))[:, 0].reshape(-1, 1)

y_pred_test = g_pred_test + H(t_test, x_test, n_test_less)
y_exact_test = exact_y(t_test, x_test)

errorL2_test = torch.sqrt((y_pred_test - y_exact_test).pow(2).sum()) / torch.sqrt(y_exact_test.pow(2).sum())
print("Relative L2 Error on 100 x 100 grid:",e(errorL2_test.item()))


g = g_pred.detach().cpu().numpy().reshape(n + 1, n + 1)
yCE = y_pred.detach().cpu().numpy().reshape(n + 1, n + 1)
y_pred_2d = yCE.reshape(n + 1, n + 1)
error_2d = np.abs(y_pred.detach().cpu().numpy().reshape(n+1, n+1) -
           exact_y(t_grid.reshape(-1, 1), x_grid.reshape(-1, 1)).cpu().numpy().reshape(n+1, n+1))

fig = plt.figure(figsize=(15,5))
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(t_grid.numpy(), x_grid.numpy(), y_pred_2d, cmap='viridis')
ax1.set_title('Approx y')

exact_y_2d = exact_y(t_grid.reshape(-1, 1), x_grid.reshape(-1, 1)).cpu().numpy().reshape(n+1, n+1)
ax2 = fig.add_subplot(132, projection='3d')
ax2.plot_surface(t_grid.numpy(), x_grid.numpy(), exact_y_2d, cmap='viridis')
ax2.set_title('Exact y')

ax3 = fig.add_subplot(133, projection='3d')
ax3.plot_surface(t_grid.numpy(), x_grid.numpy(), error_2d, cmap='inferno')
ax3.set_title('Absolute Error |y_pred - y_exact|')
plt.show()

ax4 = fig.add_subplot(122, projection='3d')
ax4.plot_surface(t_grid.numpy(), x_grid.numpy(), g, cmap='viridis')
ax4.set_title('Approx g')
plt.show()

plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Function')
plt.show()

model.eval()

rmse_test, rl2_test, abs_test, mae_test, y_test, peak_t, peak_x = test(n_test)

tt_test = torch.linspace(0, 1, n_test)
xx_test = torch.linspace(0, 1, n_test)
t_grid_test, x_grid_test = torch.meshgrid(tt_test, xx_test, indexing='ij')

y_test_np = y_test.detach().cpu().numpy().reshape(n_test, n_test)
t_grid_test_np = t_grid_test.cpu().numpy()
x_grid_test_np = x_grid_test.cpu().numpy()

y_exact_test_np = exact_y(t_grid_test.reshape(-1,1), x_grid_test.reshape(-1,1)).cpu().numpy().reshape(n_test, n_test)

error_test_np = np.abs(y_test_np - y_exact_test_np)

fig = plt.figure(figsize=(18,5))
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(t_grid_test_np, x_grid_test_np, y_test_np, rstride=1, cstride=1, cmap='viridis')
ax1.set_title('Approx y (test)')
ax1.set_xlabel('t'); ax1.set_ylabel('x')

ax2 = fig.add_subplot(132, projection='3d')
ax2.plot_surface(t_grid_test_np, x_grid_test_np, y_exact_test_np, rstride=1, cstride=1, cmap='viridis')
ax2.set_title('Exact y (test)')
ax2.set_xlabel('t'); ax2.set_ylabel('x')

ax3 = fig.add_subplot(133, projection='3d')
ax3.plot_surface(t_grid_test_np, x_grid_test_np, error_test_np, rstride=1, cstride=1, cmap='inferno')
ax3.set_title('Absolute Error |y_test - y_exact|')
ax3.set_xlabel('t'); ax3.set_ylabel('x')

plt.tight_layout()
plt.show()

plt.figure(figsize=(20,5))
plt.subplot(1,2,1)
plt.plot(train_rmse_history, label='Train RMSE')
plt.plot(test_rmse_history, label='Test RMSE')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('MSE (log scale)')
plt.title('Train/Test MSE')
plt.legend()

#plt.subplot(figsize=(10,5))
plt.subplot(1,2,2)
plt.plot(train_abs_history, label='Train MSE')
plt.plot(test_abs_history, label='Test MSE')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('ABS (log scale)')
plt.title('Train/Test MSE')
plt.legend()

plt.figure(figsize=(15,5))
plt.subplot(1,2,1)
plt.plot(train_rl2_history, label='Train RL2')
plt.plot(test_rl2_history, label='Test RL2')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Relative L2 (log scale)')
plt.title('Train/Test RL2')
plt.legend()

plt.tight_layout()
plt.show()


g = g_pred.detach().cpu().numpy().reshape(n + 1, n + 1)
yCE = y_pred.detach().cpu().numpy().reshape(n + 1, n + 1)
y_pred_2d = yCE
exact_y_2d = exact_y(t_grid.reshape(-1, 1), x_grid.reshape(-1, 1)).cpu().numpy().reshape(n+1, n+1)
error_2d = np.abs(y_pred_2d - exact_y_2d)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im1 = axes[0].imshow(y_pred_2d, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='viridis')
axes[0].set_title("Approx y")
axes[0].set_xlabel("x"); axes[0].set_ylabel("t")
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(exact_y_2d, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='viridis')
axes[1].set_title("Exact y")
axes[1].set_xlabel("x"); axes[1].set_ylabel("t")
plt.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow(error_2d, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='inferno')
axes[2].set_title("Absolute Error")
axes[2].set_xlabel("x"); axes[2].set_ylabel("t")
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()


plt.figure(figsize=(5,4))
plt.imshow(g, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='viridis')
plt.colorbar()
plt.title("Approx g")
plt.xlabel("x"); plt.ylabel("t")
plt.tight_layout()
plt.show()



y_test_np = y_test.detach().cpu().numpy().reshape(n_test, n_test)
y_exact_test_np = exact_y(t_grid_test.reshape(-1,1), x_grid_test.reshape(-1,1)).cpu().numpy().reshape(n_test, n_test)
error_test_np = np.abs(y_test_np - y_exact_test_np)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im1 = axes[0].imshow(y_test_np, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='viridis')
axes[0].set_title("Approx y (test)")
axes[0].set_xlabel("x"); axes[0].set_ylabel("t")
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(y_exact_test_np, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='viridis')
axes[1].set_title("Exact y (test)")
axes[1].set_xlabel("x"); axes[1].set_ylabel("t")
plt.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow(error_test_np, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='inferno')
axes[2].set_title("Absolute Error (test)")
axes[2].set_xlabel("x"); axes[2].set_ylabel("t")
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()


g = g_pred.detach().cpu().numpy().reshape(n + 1, n + 1)
yCE = y_pred.detach().cpu().numpy().reshape(n + 1, n + 1)
y_pred_2d = yCE
exact_y_2d = exact_y(t_grid.reshape(-1, 1), x_grid.reshape(-1, 1)).cpu().numpy().reshape(n+1, n+1)
error_2d = np.abs(y_pred_2d - exact_y_2d)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im1 = axes[0].imshow(y_pred_2d, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='viridis')
axes[0].set_title("Approx y")
axes[0].set_xlabel("x"); axes[0].set_ylabel("t")
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(exact_y_2d, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='viridis')
axes[1].set_title("Exact y")
axes[1].set_xlabel("x"); axes[1].set_ylabel("t")
plt.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow(error_2d, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='inferno')
axes[2].set_title("Absolute Error")
axes[2].set_xlabel("x"); axes[2].set_ylabel("t")
plt.colorbar(im3, ax=axes[2])
plt.tight_layout()
plt.savefig("Exact y(train)_Approx y(train)_Absolute Error(train).pdf")
plt.show()


plt.figure(figsize=(5,4))
plt.imshow(g, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='viridis')
plt.colorbar()
plt.title("Approx g")
plt.xlabel("x"); plt.ylabel("t")
plt.tight_layout()
plt.show()


y_test_np = y_test.detach().cpu().numpy().reshape(n_test, n_test)
y_exact_test_np = exact_y(t_grid_test.reshape(-1,1), x_grid_test.reshape(-1,1)).cpu().numpy().reshape(n_test, n_test)
error_test_np = np.abs(y_test_np - y_exact_test_np)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im1 = axes[0].imshow(y_test_np, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='viridis')
axes[0].set_title("Approx y (test)")
axes[0].set_xlabel("x"); axes[0].set_ylabel("t")
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(y_exact_test_np, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='viridis')
axes[1].set_title("Exact y (test)")
axes[1].set_xlabel("x"); axes[1].set_ylabel("t")
plt.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow(error_test_np, origin='lower', extent=[0,1,0,1], aspect='equal', cmap='inferno')
axes[2].set_title("Absolute Error (test)")
axes[2].set_xlabel("x"); axes[2].set_ylabel("t")
plt.colorbar(im3, ax=axes[2])
plt.tight_layout()
plt.savefig("Exact y(test)_Approx y(test)_Absolute Error(test).pdf")
plt.show()

# RL2 ___________________________________________________________
plt.figure(figsize=(15,5))
plt.plot(train_rl2_history, label='Train RL2')
plt.plot(test_rl2_history, label='Test RL2')
plt.yscale('log')
plt.xlabel('Epoch'); plt.ylabel('Relative L2 (log scale)')
plt.title('Train/Test RL2')
plt.legend()
plt.tight_layout()
plt.savefig("Train_Test_RL2.pdf")
plt.show()

# list files______________________________________________________
pdf_files = [
    "Exact y(train)_Approx y(train)_Absolute Error(train).pdf",
    "Exact y(test)_Approx y(test)_Absolute Error(test).pdf",
    "Train_Test_RL2.pdf"
]
from google.colab import files
for f in pdf_files:
    files.download(f)

